# Scouting and Draft Prediction

Two models: a classifier for whether a player-season leads to being drafted, and
a regressor for where a drafted player falls within their college class.

Both are trained only on public NCAA statistics and this package's own derived
metrics. **Read `model_card()` before quoting any of these numbers** — it
carries the limitations.

Needs the `model` extra (`pip install "ncaa_bbStats[model]"`); explanations use
`explain` for SHAP, falling back to gain-based attribution otherwise.

Covers: `scouting_report`, `predict_draft_probability`, `predict_draft_order`,
`draft_board`, `explain_prediction`, `feature_contributions`,
`predict_from_stats`, `is_draft_eligible`, `model_card`

In [1]:
from ncaa_bbStats import *

## A scouting report

In [2]:
print(scouting_report("Kade Anderson", 2025))

  Kade Anderson  |  2025  |  pitcher
  LSU  (SEC)
  Draft grade: A+   (modelled probability 99.5%)
  Projected college draft order: ~2
  Draft eligible: True (basis: drafted)
  Actual: selected #3 in round 1
------------------------------------------------------------------
  Take: the model sees a top-of-the-draft profile. Driven by age.
  Main concern: BB (Batting) (team).

  Top 5 strengths
    feature                            value      median    impact
    ^ age                             20.000      22.000     2.52%
    ^ so (pitching)                  180.000      25.000     1.28%
    ^ ip per g (pitching)              6.263       1.637     0.65%
    ^ k-bb% (pitching)                 0.301       0.081     0.28%
    ^ k% (pitching)                    0.374       0.193     0.23%

  Top 5 concerns
    feature                            value      median    impact
    v BB (Batting) (team)            360.000     249.000    -0.09%
    v q2 win pct (team)                0.909     

## The individual predictions behind it

In [3]:
for name in ["Kade Anderson", "Jamie Arnold", "Gavin Kilen", "Jac Caglianone"]:
    probability = predict_draft_probability(name, 2025)
    if probability is None:
        print(f"  {name:22s} not in the eligible population")
        continue
    order = predict_draft_order(name, 2025)
    print(f"  {name:22s} P(drafted) {probability:.1%}   "
          f"projected college order ~{order:.0f}")

  Kade Anderson          P(drafted) 99.5%   projected college order ~2
  Jamie Arnold           P(drafted) 99.6%   projected college order ~30
  Gavin Kilen            P(drafted) 98.9%   projected college order ~1
  Jac Caglianone         not in the eligible population


### Eligibility is inferred, not looked up

It comes from seasons completed and from age — and age is itself estimated for
most players. So the basis is returned alongside the answer.

In [4]:
for name in ["Kade Anderson", "Jamie Arnold", "Jac Caglianone"]:
    result = is_draft_eligible(name, 2025)
    if result is None:
        # Caglianone was drafted in 2024, so he has no 2025 college season.
        print(f"  {name:22s} no 2025 season in the data")
        continue
    eligible, basis = result
    print(f"  {name:22s} eligible={eligible}  basis={basis!r}")

  Kade Anderson          eligible=True  basis='drafted'
  Jamie Arnold           eligible=True  basis='drafted'
  Jac Caglianone         no 2025 season in the data


## What drove the prediction

SHAP where installed, gain-weighted deviation from the median otherwise. Impact
is in percentage points of draft probability.

In [5]:
explanation = explain_prediction("Kade Anderson", 2025, top_n=6)
print(f"method: {explanation['method']}")
print(f"draft probability: {explanation['draft_probability']:.1%}\n")

print("strengths:")
for row in explanation["strengths"]:
    print(f"  ^ {row['label']:28s} {row['value']:>10.3f} "
          f"(median {row['median']:>8.3f})  {row['impact']:+.2f}%")

print("\nconcerns:")
for row in explanation["concerns"]:
    print(f"  v {row['label']:28s} {row['value']:>10.3f} "
          f"(median {row['median']:>8.3f})  {row['impact']:+.2f}%")

method: shap
draft probability: 99.5%

strengths:
  ^ age                              20.000 (median   22.000)  +2.52%
  ^ so (pitching)                   180.000 (median   25.000)  +1.28%
  ^ ip per g (pitching)               6.263 (median    1.637)  +0.65%
  ^ k-bb% (pitching)                  0.301 (median    0.081)  +0.28%
  ^ k% (pitching)                     0.374 (median    0.193)  +0.23%
  ^ start share (pitching)            1.000 (median    0.077)  +0.20%

concerns:
  v BB (Batting) (team)             360.000 (median  249.000)  -0.09%
  v q2 win pct (team)                 0.909 (median    0.368)  -0.07%
  v home win pct (team)               0.854 (median    0.615)  -0.05%
  v gs (pitching)                    19.000 (median    1.000)  -0.05%
  v l (pitching)                      1.000 (median    1.000)  -0.03%
  v sv (pitching)                     0.000 (median    0.000)  -0.03%


In [6]:
# The gain fallback works with no SHAP installed
fallback = explain_prediction("Kade Anderson", 2025, method="gain", top_n=3)
print("method:", fallback["method"])
for row in fallback["strengths"]:
    print(f"  ^ {row['label']:28s} {row['impact']:+.2f}")

method: gain
  ^ so (pitching)                +26.64
  ^ q1 wins (team)               +17.04
  ^ ip (pitching)                +6.46


### Contributions that add up

`explain_prediction` returns a readable shortlist, and its impacts are
leave-one-out figures that deliberately do not sum to anything. When you need an
attribution that *does* — to stack into a waterfall, say — use
`feature_contributions`, which returns the base value and every feature's raw
contribution in the model's own units, guaranteeing
`base + sum(contributions) == prediction`.

Stage 1 works in log-odds, so apply a logistic to read a probability. Stage 2 is
already in college draft order units. Requires the `explain` extra; there is no
gain fallback, because gain has no base value.

In [7]:
from math import exp

stage1 = feature_contributions("Kade Anderson", 2025, stage=1)
total = stage1["base"] + sum(c["contribution"] for c in stage1["contributions"])
print(f"units: {stage1['units']}   features: {len(stage1['contributions'])}")
print(f"base {stage1['base']:+.3f} -> prediction {stage1['prediction']:+.3f} "
      f"(sum checks: {abs(total - stage1['prediction']) < 1e-9})")
print(f"as a probability: {1 / (1 + exp(-stage1['base'])):.1%} -> "
      f"{1 / (1 + exp(-stage1['prediction'])):.1%}\n")

for row in stage1["contributions"][:6]:
    value = "not supplied" if row["value"] is None else f"{row['value']:.3f}"
    print(f"  {row['label']:28s} {value:>12s}  {row['contribution']:+.3f}")

stage2 = feature_contributions("Kade Anderson", 2025, stage=2)
print(f"\nstage 2 ({stage2['units']}): {stage2['base']:.1f} -> "
      f"{stage2['prediction']:.1f}")

units: log-odds   features: 147
base -2.143 -> prediction +5.396 (sum checks: True)
as a probability: 10.5% -> 99.5%

  age                                20.000  +1.910
  so (pitching)                     180.000  +1.359
  ip per g (pitching)                 6.263  +0.899
  k-bb% (pitching)                    0.301  +0.487
  k% (pitching)                       0.374  +0.408
  start share (pitching)              1.000  +0.363

stage 2 (draft order): 203.1 -> 2.2


In [8]:
# The same function explains a stat line that was never in the data: hand it the
# feature_row that predict_from_stats scored, so the explanation and the number
# come from one row rather than two.
scored = predict_from_stats(
    "pitcher", 21,
    {"era_pitch": 2.40, "so_pitch": 130, "bb_pitch": 25, "ip_pitch": 95.0},
    team="LSU", season=2025,
)
custom = feature_contributions(features=scored["feature_row"], stage=1)
print(f"P(drafted) {scored['draft_probability']:.1%}, "
      f"from {len(scored['supplied_features'])} supplied statistics")
for row in custom["contributions"][:5]:
    value = "not supplied" if row["value"] is None else f"{row['value']:.3f}"
    print(f"  {row['label']:28s} {value:>12s}  {row['contribution']:+.3f}")

P(drafted) 80.3%, from 4 supplied statistics
  so (pitching)                     130.000  +1.457
  age                                21.000  +0.759
  q1 wins (team)                     19.000  +0.393
  budget pct (team)                   0.999  +0.285
  rpi rank (team)                     4.000  +0.204


## A whole season, ranked

`draft_board` scores every eligible player. Comparing against `actual_pick`
shows where the model agreed with the draft and where it did not.

In [9]:
board = draft_board(2026, n=15)   # 2026 is the model's held-out season
print(f"  {'#':>3} {'player':24s} {'team':6s} {'P(draft)':>9s} {'grade':>6s} "
      f"{'proj':>6s} {'actual':>7s}")
for row in board:
    projected = f"{row['predicted_order']:.0f}" if row["predicted_order"] else "-"
    actual = f"#{row['actual_pick']}" if row["actual_pick"] else "undrafted"
    print(f"  {row['rank']:>3} {row['name']:24s} {row['team']:6s} "
          f"{row['draft_probability']:>9.3f} {row['draft_grade']:>6s} "
          f"{projected:>6s} {actual:>7s}")

    # player                   team    P(draft)  grade   proj  actual
    1 Jarren Advincula         GT         0.998     A+     43     #45
    2 Charlie West             CONN       0.996     A+    128 undrafted
    3 Vahn Lackey              GT         0.996     A+     14      #3
    4 Drew Burress             GT         0.995     A+     40      #8
    5 Mason Edwards            USC        0.995     A+      9     #47
    6 Jack Radel               ND         0.994     A+     80     #28
    7 Jake Schaffner           UNC        0.993     A+     29     #20
    8 Owen Hull                UNC        0.992     A+     55     #67
    9 Wes Mendes               FSU        0.992     A+     44     #57
   10 Evan Dempsey             FGCU       0.991     A+     98     #69
   11 Daniel Jackson           UGA        0.991     A+     59     #37
   12 Cole Carlon              ASU        0.990     A+      1     #39
   13 Logan Reddemann          UCLA       0.988     A+     74     #38
   14 Gage Peterso

In [10]:
# How much of the top of the board was actually drafted?
top50 = draft_board(2026, n=50)
hit = sum(1 for row in top50 if row["actual_pick"])
print(f"{hit} of the top 50 were drafted ({hit / 50:.0%})")

# Draft position is suppressed below 25%: the order model is trained only on
# drafted players, so applying it lower down would be extrapolation.
low = [r for r in draft_board(2026, n=2000) if r["draft_probability"] < 0.25]
print(f"\n{len(low)} players below the 25% threshold; "
      f"all have predicted_order suppressed: "
      f"{all(r['predicted_order'] is None for r in low)}")

48 of the top 50 were drafted (96%)



1429 players below the 25% threshold; all have predicted_order suppressed: True


## Scoring a line that is not in the data

Supply as much or as little as you have. Unspecified statistics stay missing,
which the models handle natively, and the result reports how much was imputed.

In [11]:
result = predict_from_stats(
    "pitcher", age=21,
    stats={"era": 2.40, "so": 130, "bb": 25, "ip": 95.0,
           "h": 68, "hr": 5, "g": 16, "gs": 16, "tbf": 370},
    team="LSU", name="Prospect A",   # season defaults to the most recent
)
print(result["report"])

  Prospect A  |  pitcher, age 21  |  2026 context  (LSU)
  Draft grade: A+   (modelled probability 92.5%)
  Projected college draft order: ~60
------------------------------------------------------------------
  Supplied 9 statistics; 63 left unset (85%).
  Confidence: low.
  Team context imputed from the 2026 median (1 fields).


In [12]:
print("supplied:", result["supplied_features"])
print("confidence:", result["confidence"])
print("draft probability:", round(result["draft_probability"], 4))
print("predicted order:", result["predicted_order"])

supplied: ['bb_pitch', 'era_pitch', 'g_pitch', 'gs_pitch', 'h_pitch', 'hr_pitch', 'ip_pitch', 'so_pitch', 'tbf_pitch']
confidence: low
draft probability: 0.9251
predicted order: 59.788047790527344


In [13]:
# A weaker line, with no team given -- context falls back to the season median
weak = predict_from_stats(
    "batter", age=19,
    stats={"avg": 0.240, "hr": 1, "pa": 90, "ab": 80},
    name="Prospect B",   # no team either: context falls back to the league median
)
print(weak["report"])

  Prospect B  |  batter, age 19  |  2026 context  (league median)
  Draft grade: D+   (modelled probability 6.8%)
  Draft order suppressed: probability 6.8% is below 25%.
------------------------------------------------------------------
  Supplied 4 statistics; 68 left unset (92%).
  Confidence: low.
  Team context imputed from the 2026 median (73 fields).


In [14]:
# The model responds to the input: same role and age, different production
strong = predict_from_stats("pitcher", 21,
    {"era": 1.80, "so": 150, "bb": 15, "ip": 100.0, "h": 60, "hr": 3,
     "g": 16, "gs": 16}, team="LSU")
poor = predict_from_stats("pitcher", 21,
    {"era": 7.50, "so": 12, "bb": 20, "ip": 18.0, "h": 30, "hr": 6,
     "g": 9, "gs": 1}, team="LSU")

print(f"  strong line: {strong['draft_probability']:.1%}  "
      f"grade {strong['draft_grade']}")
print(f"  poor line  : {poor['draft_probability']:.1%}  "
      f"grade {poor['draft_grade']}")

  strong line: 92.7%  grade A+
  poor line  : 28.1%  grade C


## The model card

Published as a function rather than a documentation footnote, so the
limitations travel with the predictions.

In [15]:
card = model_card()
print(f"version    : {card['model_version']}")
print(f"trained on : {card['train_years']}")
print(f"tested on  : {card['test_year']}")
print(f"eligibility: {card['eligibility']}")

print(f"\nstage 1 (drafted or not): {card['stage1']['metrics']}")
print(f"stage 2 (draft order)   : {card['stage2']['metrics']}")

version    : v7-public-2026.1
trained on : [2021, 2022, 2023, 2024, 2025]
tested on  : 2026
eligibility: {'age': 21, 'seasons': 3, 'unknown_treated_as_eligible': False}

stage 1 (drafted or not): {'base_rate': 0.0776, 'n_test': 5695, 'n_train': 19629, 'pr_auc': 0.703, 'roc_auc': 0.9567}
stage 2 (draft order)   : {'mae': 78.12, 'n_test': 442, 'n_train': 2024, 'p_value': 7.599972344174245e-54, 'spearman': 0.6473}


In [16]:
print("Stage 3 is deliberately not shipped:")
print(" ", card["stage3"]["reason"])

Stage 3 is deliberately not shipped:
  A bonus/slot ratio model was attempted and scored a rank correlation of 0.003 on held-out data -- indistinguishable from noise. Slot values are published facts and are available via draft_detail_utils.slot_value().


In [17]:
print("Limitations:")
for i, limitation in enumerate(card["limitations"], 1):
    print(f"\n  {i}. {limitation}")

Limitations:

  1. Stage 1 precision depends on the base rate of the population it is applied to. On the held-out season roughly 7% of eligible players were drafted; applied to a pre-screened shortlist, precision is higher, and applied to every player in the country, lower.

  2. Draft eligibility is inferred from seasons completed and from age, and age is itself estimated for most players. is_draft_eligible() returns the basis so the inference is visible.

  3. The order model is trained only on players who were drafted. Applying it below a 25% draft probability is extrapolation, and it is suppressed there.

  4. No third stage. A bonus/slot ratio model scored a rank correlation of 0.003 on held-out data and is not shipped; slot values are published facts, available via draft_detail_utils.slot_value().

  5. Trained on 2021-2025 and tested on 2026, a single held-out season. These are not cross-validated estimates, and one season is a small sample: 2026 scores lower than 2025 did, whic

In [18]:
reference = card["reference_implementation"]
print("For comparison only -- NOT this package's numbers:")
print(f"  {reference['note']}\n")
print(f"  reference PR-AUC   {reference['stage1_pr_auc']}   "
      f"vs this package {card['stage1']['metrics']['pr_auc']}")
print(f"  reference ROC-AUC  {reference['stage1_roc_auc']}   "
      f"vs this package {card['stage1']['metrics']['roc_auc']}")
print(f"  reference Spearman {reference['stage2_spearman']}   "
      f"vs this package {card['stage2']['metrics']['spearman']}")

For comparison only -- NOT this package's numbers:
  V7 is the published research model this one derives from. It used proprietary third-party metrics as features, a different label set and population, its own hyperparameters, and a 2026 test year. The numbers below are for orientation only. They are NOT this model's performance, and because features, labels, population and settings all differ, the gap between the two cannot be attributed to any one of them. See model_card()['lineage'].

  reference PR-AUC   0.725   vs this package 0.703
  reference ROC-AUC  0.949   vs this package 0.9567
  reference Spearman 0.653   vs this package 0.6473
